# 📘 Guía de Estudio 3: Pérdida Contrastiva InfoNCE y Alineación de Espacios

En esta tercera guía interactiva, aprenderás:
1. Qué significa "alinear representaciones" visuales con representaciones de texto.
2. Cómo funciona la **Pérdida InfoNCE (Contrastive Loss)** matemáticamente.
3. Cómo calcular y optimizar esta pérdida usando PyTorch.

---

### 1. El Concepto de Alineación Semántica

Imagínate que tenemos dos mundos:
- **Mundo Visual:** Los embeddings que genera nuestro proyector de landmarks de señas.
- **Mundo Textual:** Los embeddings que genera el LLM para las palabras escritas (ej: "HOLA", "ADIOS").

Inicialmente, el proyector genera vectores aleatorios porque no está entrenado. Por lo tanto, el vector del video de "HOLA" y la palabra de texto "HOLA" están muy lejos en el espacio matemático.

La pérdida contrastiva **InfoNCE**:
1. Toma un par de ejemplo coincidente (Video de HOLA, Texto de HOLA) -> Par positivo.
2. Toma pares no coincidentes (Video de HOLA, Texto de ADIOS) -> Par negativo.
3. Modifica los pesos del proyector para **maximizar la similitud coseno** del par positivo, y **minimizar** la similitud de los pares negativos.

### 2. Similitud Coseno e InfoNCE

La similitud coseno entre dos vectores normalizados $\mathbf{v}$ y $\mathbf{t}$ es:

$$\text{sim}(\mathbf{v}, \mathbf{t}) = \frac{\mathbf{v} \cdot \mathbf{t}}{\|\mathbf{v}\| \|\mathbf{t}\|}$$

In [ ]:
import torch
import torch.nn.functional as F

# Simulación de 2 muestras de video y 2 muestras de texto en el batch (Dimensión = 2560)
B, D = 2, 2560
visual_embeds = torch.randn(B, D)
text_embeds = torch.randn(B, D)

# Normalización L2
v_norm = F.normalize(visual_embeds, dim=-1)
t_norm = F.normalize(text_embeds, dim=-1)

# Multiplicación matricial para obtener similitudes entre todos
similarity_matrix = torch.matmul(v_norm, t_norm.transpose(0, 1))
print("Matriz de similitud original (aleatoria):\n", similarity_matrix)

### 📝 Ejercicio 3: Implementar la pérdida InfoNCE básica

Implementa la función de pérdida InfoNCE y observa cómo varía al cambiar los vectores para que sean más similares o diferentes.

In [ ]:
def calcular_infonce(visual, text, temperature=0.07):
    # TODO:
    # 1. Normaliza los vectores visual y text usando F.normalize
    # 2. Calcula la matriz de similitud usando multiplicación matricial y dividiendo por la temperatura
    # 3. Crea las etiquetas target (deben ser torch.arange(B))
    # 4. Calcula la pérdida de cross_entropy en ambos ejes (visual a texto, y texto a visual) y saca el promedio
    pass

# --- Validación ---
loss = calcular_infonce(visual_embeds, text_embeds)
if loss is not None:
    print(f"[+] Pérdida calculada: {loss.item():.4f}")
else:
    print("[-] Por favor, completa la función primero.")

### Solución sugerida para validar la pérdida:
```python
def calcular_infonce(visual, text, temperature=0.07):
    v_norm = F.normalize(visual, dim=-1)
    t_norm = F.normalize(text, dim=-1)
    logits = torch.matmul(v_norm, t_norm.transpose(0, 1)) / temperature
    labels = torch.arange(logits.shape[0], device=logits.device)
    loss_v = F.cross_entropy(logits, labels)
    loss_t = F.cross_entropy(logits.transpose(0, 1), labels)
    return (loss_v + loss_t) / 2.0
```